## Overview
<p>
The scenario for this dataset is, there are a group of hospitals in different sizes small, medium and large.There are normal ways in which sales are created just organically and sales made through marketting tactics such as phone, email, flyers and a direct sales visit.
    </p>
    
In this project we will answer: 
<p>What is the impact of each marketing strategy and sales visit on Sales (Amount Collected)?</p>
Is the same strategy valid for all the different client types ?

In [1]:
pip install statsmodels

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install mysql-connector-python

Note: you may need to restart the kernel to use updated packages.


In [3]:
#import libraries
import pandas as pd
import numpy as np
import seaborn as sns
import scipy.stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
import os

#### Data Loading

In [4]:
data=pd.read_csv("C:/Users/melvi/Downloads/Campaign-Data.csv")
data.columns

Index(['Client ID', 'Client Type', 'Number of Customers', 'Montly Target',
       'Zip Code', 'Calendardate', 'Amount Collected', 'Unit Sold',
       'Campaign (Email)', 'Campaign (Flyer)', 'Campaign (Phone)',
       'Sales Contact 1', 'Sales Contact 2', 'Sales Contact 3',
       'Sales Contact 4', 'Sales Contact 5', 'Number of Competition'],
      dtype='object')

In [5]:
data.head()

,Client ID,Client Type,Number of Customers,Montly Target,Zip Code,Calendardate,Amount Collected,Unit Sold,Campaign (Email),Campaign (Flyer),Campaign (Phone),Sales Contact 1,Sales Contact 2,Sales Contact 3,Sales Contact 4,Sales Contact 5,Number of Competition
0,ID-987275,Medium Facility,2800,125,1003,16-01-2014,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Low
1,ID-987275,Medium Facility,2800,125,1003,16-02-2014,3409460,24,0.0,0.0,0.0,0.0,0.0,0.0,0.0,322500.0,Low
2,ID-987275,Medium Facility,2800,125,1003,18-03-2014,10228384,75,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Low
3,ID-987275,Medium Facility,2800,125,1003,18-04-2014,17047304,123,0.0,0.0,0.0,0.0,3547500.0,1290000.0,0.0,0.0,Low
4,ID-987275,Medium Facility,2800,125,1003,19-05-2014,23866224,171,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Low


#### Exploratory Data Analysis

In [6]:
# Creation of Additional Features 
data['Calendardate']=pd.to_datetime(data['Calendardate'])
data['Calendar_Month']=data['Calendardate'].dt.month
data['Calendar_Year']=data['Calendardate'].dt.year

C:\Users\melvi\anaconda3\lib\site-packages\pandas\core\tools\datetimes.py:1047: UserWarning: Parsing '16-01-2014' in DD/MM/YYYY format. Provide format or specify infer_datetime_format=True for consistent parsing.
  cache_array = _maybe_cache(arg, format, cache, convert_listlike)
C:\Users\melvi\anaconda3\lib\site-packages\pandas\core\tools\datetimes.py:1047: UserWarning: Parsing '16-02-2014' in DD/MM/YYYY format. Provide format or specify infer_datetime_format=True for consistent parsing.
  cache_array = _maybe_cache(arg, format, cache, convert_listlike)
C:\Users\melvi\anaconda3\lib\site-packages\pandas\core\tools\datetimes.py:1047: UserWarning: Parsing '18-03-2014' in DD/MM/YYYY format. Provide format or specify infer_datetime_format=True for consistent parsing.
  cache_array = _maybe_cache(arg, format, cache, convert_listlike)
C:\Users\melvi\anaconda3\lib\site-packages\pandas\core\tools\datetimes.py:1047: UserWarning: Parsing '18-04-2014' in DD/MM/YYYY format. Provide format or specif

In [7]:
data['Client Type'].value_counts(normalize=True)

Large Facility      0.459677
Small Facility      0.282258
Medium Facility     0.169355
Private Facility    0.088710
Name: Client Type, dtype: float64

In [8]:
data.groupby('Number of Competition').mean()

,Number of Customers,Montly Target,Zip Code,Amount Collected,Unit Sold,Campaign (Email),Campaign (Flyer),Campaign (Phone),Sales Contact 1,Sales Contact 2,Sales Contact 3,Sales Contact 4,Sales Contact 5,Calendar_Month,Calendar_Year
Number of Competition,,,,,,,,,,,,,,,
High,1456.935484,75.080645,1003.0,2.974789e+07,213.127016,105398.938508,994046.717540,45198.036895,146945.564516,2.685333e+06,1.786754e+06,72172.379032,8452.620968,10.5,2015.0
Low,1456.935484,75.080645,1003.0,1.445570e+07,103.132258,150862.165766,623692.979839,26693.304194,128219.758065,1.890916e+06,1.883634e+06,70481.854839,15864.919355,5.7,2014.4


In [10]:
data.corr()[['Amount Collected']]

,Amount Collected
Number of Customers,0.607496
Montly Target,0.608204
Zip Code,NaN
Amount Collected,1.000000
Unit Sold,0.997515
Campaign (Email),0.248235
Campaign (Flyer),0.444337
Campaign (Phone),0.034858
Sales Contact 1,0.277478
Sales Contact 2,0.552112


#### Correlation Analysis

In [11]:
# Consolidated Strategy for Targeting
cm = sns.light_palette("green", as_cmap=True)
correlation_analysis=pd.DataFrame(data[['Amount Collected',
'Campaign (Email)', 'Campaign (Flyer)', 'Campaign (Phone)',
       'Sales Contact 1', 'Sales Contact 2', 'Sales Contact 3',
       'Sales Contact 4', 'Sales Contact 5']].corr()['Amount Collected']).reset_index()
correlation_analysis.columns=['Impacting Variable','Degree of Linear Impact (Correlation)']
correlation_analysis=correlation_analysis[correlation_analysis['Impacting Variable']!='Amount Collected']
correlation_analysis=correlation_analysis.sort_values('Degree of Linear Impact (Correlation)',ascending=False)
correlation_analysis.style.background_gradient(cmap=cm).set_precision(2)

C:\Users\melvi\AppData\Local\Temp\ipykernel_3516\3909605169.py:10: FutureWarning: this method is deprecated in favour of `Styler.format(precision=..)`
  correlation_analysis.style.background_gradient(cmap=cm).set_precision(2)


,Impacting Variable,Degree of Linear Impact (Correlation)
5,Sales Contact 2,0.55
2,Campaign (Flyer),0.44
6,Sales Contact 3,0.36
4,Sales Contact 1,0.28
1,Campaign (Email),0.25
7,Sales Contact 4,0.24
8,Sales Contact 5,0.10
3,Campaign (Phone),0.03


#### Market Strategy Impact on Sales for different account types

In [12]:
cm = sns.light_palette("green", as_cmap=True)
correlation_analysis=pd.DataFrame(data.groupby('Client Type')[['Amount Collected',
       'Campaign (Email)', 'Campaign (Flyer)', 'Campaign (Phone)',
       'Sales Contact 1', 'Sales Contact 2', 'Sales Contact 3',
       'Sales Contact 4', 'Sales Contact 5']].corr()['Amount Collected']).reset_index()
correlation_analysis=correlation_analysis.sort_values(['Client Type','Amount Collected'],ascending=False)
correlation_analysis.columns=['Acc Type','Variable Impact on Sales','Impact']
correlation_analysis=correlation_analysis[correlation_analysis['Variable Impact on Sales']!='Amount Collected'].reset_index(drop=True)
correlation_analysis.style.background_gradient(cmap=cm).set_precision(2)

C:\Users\melvi\AppData\Local\Temp\ipykernel_3516\1544892028.py:9: FutureWarning: this method is deprecated in favour of `Styler.format(precision=..)`
  correlation_analysis.style.background_gradient(cmap=cm).set_precision(2)


,Acc Type,Variable Impact on Sales,Impact
0,Small Facility,Sales Contact 2,0.22
1,Small Facility,Sales Contact 3,0.07
2,Small Facility,Campaign (Email),0.06
3,Small Facility,Campaign (Flyer),0.04
4,Small Facility,Sales Contact 4,0.02
5,Small Facility,Sales Contact 5,0.00
6,Small Facility,Sales Contact 1,-0.02
7,Small Facility,Campaign (Phone),nan
8,Private Facility,Sales Contact 2,0.57
9,Private Facility,Campaign (Flyer),0.28


#### Regression Analysis in Market Sales and Strategies

In [13]:
data.columns=[mystring.replace(" ", "_") for mystring in data.columns]
data.columns=[mystring.replace("(", "") for mystring in data.columns]
data.columns=[mystring.replace(")", "") for mystring in data.columns]
# using the OLS formula: y = mx + b
results = smf.ols('Amount_Collected ~ Campaign_Email+Campaign_Flyer+Campaign_Phone+\
       Sales_Contact_1 + Sales_Contact_2 + Sales_Contact_3+Sales_Contact_4 + Sales_Contact_5',data=data).fit()
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:       Amount_Collected   R-squared:                       0.480
Model:                            OLS   Adj. R-squared:                  0.478
Method:                 Least Squares   F-statistic:                     342.1
Date:                Wed, 01 Mar 2023   Prob (F-statistic):               0.00
Time:                        21:17:47   Log-Likelihood:                -54512.
No. Observations:                2976   AIC:                         1.090e+05
Df Residuals:                    2967   BIC:                         1.091e+05
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept        1.481e+06   5.12e+05     

In [14]:
df = pd.read_html(results.summary().tables[1].as_html(),header=0,index_col=0)[0]

In [15]:
# Eliminating the high p value
df=df.reset_index()
df=df[df['P>|t|']<0.05][['index','coef']]
df

,index,coef
0,Intercept,1.481000e+06
2,Campaign_Flyer,3.337600e+00
4,Sales_Contact_1,4.236800e+00
5,Sales_Contact_2,3.638200e+00
6,Sales_Contact_3,2.343200e+00
7,Sales_Contact_4,1.094780e+01


#### Regression Analysis for different account types

In [16]:
consolidated_summary=pd.DataFrame()
for acctype in list(set(list(data['Client_Type']))):
    print(acctype)
    temp_data=data[data['Client_Type']==acctype].copy()
    results = smf.ols('Amount_Collected ~ Campaign_Email+Campaign_Flyer+Campaign_Phone+\
       Sales_Contact_1 + Sales_Contact_2 + Sales_Contact_3+Sales_Contact_4 + Sales_Contact_5', data=temp_data).fit()
    df = pd.read_html(results.summary().tables[1].as_html(),header=0,index_col=0)[0].reset_index()
    df=df[df['P>|t|']<0.05][['index','coef']]
    df.columns=['Variable','Coefficent (Impact)']
    df['Account Type']=acctype
    df=df.sort_values('Coefficent (Impact)',ascending=False)
    df=df[df['Variable']!='Intercept']
    consolidated_summary=consolidated_summary.append(df)
    print(results.summary())

Small Facility 
                            OLS Regression Results                            
Dep. Variable:       Amount_Collected   R-squared:                       0.054
Model:                            OLS   Adj. R-squared:                  0.046
Method:                 Least Squares   F-statistic:                     6.829
Date:                Wed, 01 Mar 2023   Prob (F-statistic):           6.63e-08
Time:                        21:17:49   Log-Likelihood:                -14125.
No. Observations:                 840   AIC:                         2.827e+04
Df Residuals:                     832   BIC:                         2.830e+04
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept        8.789e+05

C:\Users\melvi\AppData\Local\Temp\ipykernel_3516\3385269189.py:13: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  consolidated_summary=consolidated_summary.append(df)
C:\Users\melvi\AppData\Local\Temp\ipykernel_3516\3385269189.py:13: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  consolidated_summary=consolidated_summary.append(df)
C:\Users\melvi\AppData\Local\Temp\ipykernel_3516\3385269189.py:13: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  consolidated_summary=consolidated_summary.append(df)


                            OLS Regression Results                            
Dep. Variable:       Amount_Collected   R-squared:                       0.371
Model:                            OLS   Adj. R-squared:                  0.367
Method:                 Least Squares   F-statistic:                     100.2
Date:                Wed, 01 Mar 2023   Prob (F-statistic):          4.64e-131
Time:                        21:17:49   Log-Likelihood:                -25075.
No. Observations:                1368   AIC:                         5.017e+04
Df Residuals:                    1359   BIC:                         5.021e+04
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept        2.812e+06   9.29e+05     

C:\Users\melvi\AppData\Local\Temp\ipykernel_3516\3385269189.py:13: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  consolidated_summary=consolidated_summary.append(df)


#### Visualization

In [17]:
consolidated_summary

,Variable,Coefficent (Impact),Account Type
5,Sales_Contact_2,8.101000e-01,Small Facility
3,Campaign_Phone,-2.603000e-07,Small Facility
4,Sales_Contact_1,1.167310e+01,Large Facility
7,Sales_Contact_4,1.061450e+01,Large Facility
5,Sales_Contact_2,4.003100e+00,Large Facility
2,Campaign_Flyer,2.720400e+00,Large Facility
6,Sales_Contact_3,2.031600e+00,Large Facility
3,Campaign_Phone,-3.536100e+00,Large Facility
5,Sales_Contact_2,6.622300e+00,Private Facility
2,Campaign_Flyer,4.105900e+00,Medium Facility


In [18]:
consolidated_summary.reset_index(inplace=True)
consolidated_summary.drop('index',inplace=True,axis=1)

In [19]:
#Heatmap
consolidated_summary.columns = ['Variable','Return on Investment','Account Type']
consolidated_summary['Return on Investment']= consolidated_summary['Return on Investment'].apply(lambda x: round(x,1))
consolidated_summary.style.background_gradient(cmap='RdYlGn')

,Variable,Return on Investment,Account Type
0,Sales_Contact_2,0.800000,Small Facility
1,Campaign_Phone,-0.000000,Small Facility
2,Sales_Contact_1,11.700000,Large Facility
3,Sales_Contact_4,10.600000,Large Facility
4,Sales_Contact_2,4.000000,Large Facility
5,Campaign_Flyer,2.700000,Large Facility
6,Sales_Contact_3,2.000000,Large Facility
7,Campaign_Phone,-3.500000,Large Facility
8,Sales_Contact_2,6.600000,Private Facility
9,Campaign_Flyer,4.100000,Medium Facility


In [20]:
# including a $ into the ROI for further analysis in Power Bi
def format(x):
        return "${:.1f}".format(x)
consolidated_summary['Return on Investment']  = consolidated_summary['Return on Investment'].apply(format)

In [21]:
consolidated_summary.columns = ['Variable','Return on Investment','Account Type']
consolidated_summary.style.background_gradient(cmap='RdYlGn')

,Variable,Return on Investment,Account Type
0,Sales_Contact_2,$0.8,Small Facility
1,Campaign_Phone,$-0.0,Small Facility
2,Sales_Contact_1,$11.7,Large Facility
3,Sales_Contact_4,$10.6,Large Facility
4,Sales_Contact_2,$4.0,Large Facility
5,Campaign_Flyer,$2.7,Large Facility
6,Sales_Contact_3,$2.0,Large Facility
7,Campaign_Phone,$-3.5,Large Facility
8,Sales_Contact_2,$6.6,Private Facility
9,Campaign_Flyer,$4.1,Medium Facility


In [22]:
# saved the dataframe into a CSV file and downloaded it
# consolidated_summary.to_csv('consolidated_summary.csv')

### Import Dataframe to MySql

In [23]:
import mysql.connector

In [24]:
# create a connection to MySQL
conn =  mysql.connector.connect(
  host=os.getenv('HOST'),
  user=os.getenv('USER'),
  password=os.getenv('PASS'),
  database=os.getenv('DB')
) 

In [25]:
# create a cursor
cursor = conn.cursor()

In [27]:
# create the table in MySQL
table_name = 'summary'
query = f"CREATE TABLE {table_name} (Variable VARCHAR(255), Return_in_Investment VARCHAR(255), Account_Type VARCHAR(255))"
cursor.execute(query)

# insert the data from the dataframe into the table
for row in consolidated_summary.itertuples(index=False):
    query = f"INSERT INTO {table_name} (Variable, Return_in_Investment, Account_Type) VALUES (%s, %s, %s)"
    cursor.execute(query, row)


In [28]:
# commit the changes and close the connection
conn.commit()
conn.close()